# KMeans Custom Algorithm — Step-by-Step Visual Guide

This notebook visually walks through **every step** of our custom KMeans implementation.

The algorithm has two major phases:

| Phase | Name | Description |
|-------|------|-------------|
| **A** | **ESPI Initialization** (Range-Based Centroid Seeding) | Deterministic centroid placement along the dimension with highest variance. O(n*d) — no random restarts needed. |
| **B** | **Drift-Based Iterative Refinement** | Standard centroid-update loop accelerated by a triangle-inequality trick that skips stable interior points. |

We demonstrate on three datasets:
1. **Iris** (4 features, 150 samples) — classic benchmark, k=3
2. **AI Index** (numeric features, ~60 countries) — higher-dimensional policy data, k=3
3. **Earthquakes** (magnitude, depth, latitude, longitude) — geospatial data, k=3

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from kmeans_custom import KMeansCustom

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8

In [ ]:
# ── Load and preprocess all 3 datasets ──────────────────────────────

# 1. Iris
iris_df = pd.read_csv('data/iris.csv')
iris_features = iris_df.drop(columns=['species'])
iris_scaled = StandardScaler().fit_transform(iris_features)
iris_feature_names = list(iris_features.columns)

# 2. AI Index
ai_df = pd.read_csv('data/ai_index.csv')
ai_numeric = ai_df.select_dtypes(include='number').dropna()
ai_scaled = StandardScaler().fit_transform(ai_numeric)
ai_feature_names = list(ai_numeric.columns)

# 3. Earthquakes
eq_df = pd.read_csv('data/earthquakes.csv')
eq_cols = ['magnitude', 'depth', 'latitude', 'longitude']
eq_subset = eq_df[eq_cols].dropna()
eq_scaled = StandardScaler().fit_transform(eq_subset)
eq_feature_names = eq_cols

# Store in a dict for easy iteration
datasets = {
    'Iris':        {'data': iris_scaled, 'names': iris_feature_names},
    'AI Index':    {'data': ai_scaled,   'names': ai_feature_names},
    'Earthquakes': {'data': eq_scaled,   'names': eq_feature_names},
}

for name, d in datasets.items():
    print(f"{name:15s}  shape = {d['data'].shape}  features = {d['names']}")

## Phase A: ESPI Initialization (Range-Based Centroid Seeding)

Instead of random or k-means++ initialization, ESPI spreads centroids **evenly along the dimension with the highest variance**.

**Four sub-steps:**

1. **Find the dominant dimension** — compute variance for every feature; the one with the largest variance is the most discriminative axis.
2. **Compute the range** `[min, max]` of the data along that dominant dimension.
3. **Divide into k equal segments** — split the range into k intervals of equal width.
4. **Place centroids at segment midpoints** — each centroid sits at the center of its segment on the dominant axis; all other dimensions are set to the **global mean**.

**Time complexity:** O(n * d) — a single pass to compute means and variances, then O(k * d) to build centroids.

**Guarantees:**
- No two centroids can coincide (segments are non-overlapping).
- Maximum spread along the most informative axis.
- Fully deterministic — no randomness.

### Step 1: Identify the Dominant Dimension

The dimension with the **highest variance** has the most discriminative spread across the data. This is the axis along which centroids will be spaced. Below we visualize the per-feature variance for each dataset and highlight the dominant dimension in red.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, d) in zip(axes, datasets.items()):
    data = d['data']
    names = d['names']
    variances = np.var(data, axis=0)
    dominant = np.argmax(variances)
    colors = ['#d62728' if i == dominant else '#1f77b4' for i in range(len(variances))]

    bars = ax.bar(range(len(variances)), variances, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Variance')
    ax.set_title(f"{name}\nDominant: {names[dominant]} (var={variances[dominant]:.3f})")

fig.suptitle('Step 1: Per-Feature Variance — Dominant Dimension in Red', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### Steps 2 & 3: Divide the Range and Place Centroids

We take **[min, max]** of the dominant dimension and split it into **k segments of equal width**:

$$\text{segment\_width} = \frac{\max - \min}{k}$$

Each centroid is placed at the **midpoint** of its segment on the dominant axis:

$$c_j[\text{dominant}] = \min + (j + 0.5) \times \text{segment\_width}, \quad j = 0, 1, \ldots, k-1$$

For **all other dimensions**, the centroid gets the **global mean** value. This ensures centroids are well-separated along the axis of greatest variance while remaining centered in the data cloud on every other axis.

In [ ]:
# ── 1D strip-plot visualization for Iris (k=3) ──────────────────────
k = 3
data_iris = datasets['Iris']['data']
names_iris = datasets['Iris']['names']

variances_iris = np.var(data_iris, axis=0)
dom_dim = int(np.argmax(variances_iris))
dom_vals = data_iris[:, dom_dim]
d_min, d_max = dom_vals.min(), dom_vals.max()
seg_w = (d_max - d_min) / k
global_mean = data_iris.mean(axis=0)

# Build centroid positions on dominant axis
midpoints = [d_min + (j + 0.5) * seg_w for j in range(k)]

fig, ax = plt.subplots(figsize=(12, 3))

# Colored bands for each segment
band_colors = ['#fce4ec', '#e8f5e9', '#e3f2fd']
for j in range(k):
    lo = d_min + j * seg_w
    hi = lo + seg_w
    ax.axvspan(lo, hi, alpha=0.4, color=band_colors[j], label=f'Segment {j}')
    ax.axvline(lo, color='gray', linestyle='--', linewidth=0.8)
ax.axvline(d_max, color='gray', linestyle='--', linewidth=0.8)

# Strip plot of all data points on the dominant axis
jitter = np.random.default_rng(42).uniform(-0.15, 0.15, size=len(dom_vals))
ax.scatter(dom_vals, jitter, s=15, alpha=0.6, color='black', zorder=3, label='Data points')

# Centroid midpoints
for j, mp in enumerate(midpoints):
    ax.scatter(mp, 0, marker='*', s=250, color=plt.cm.Set1(j), edgecolors='black',
               linewidths=1, zorder=5)

ax.set_xlabel(f'Dominant dimension: {names_iris[dom_dim]} (standardized)')
ax.set_yticks([])
ax.set_title(f'Iris (k={k}): Range [{d_min:.2f}, {d_max:.2f}] divided into {k} equal segments\n'
             f'Segment width = {seg_w:.3f} | Stars = centroid positions on dominant axis')
ax.legend(loc='upper right', fontsize=7)
plt.tight_layout()
plt.show()

# Print full centroid coordinates
print("\nInitial centroid coordinates (all dimensions):")
for j in range(k):
    centroid = global_mean.copy()
    centroid[dom_dim] = midpoints[j]
    coords = ', '.join(f"{names_iris[i]}={centroid[i]:.3f}" for i in range(len(names_iris)))
    print(f"  C{j}: {coords}")

In [ ]:
# ── PCA 2D projection of ESPI centroids on all 3 datasets ──────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, (name, d) in zip(axes, datasets.items()):
    data = d['data']
    fnames = d['names']
    variances = np.var(data, axis=0)
    dd = int(np.argmax(variances))

    # Build ESPI centroids
    km_tmp = KMeansCustom(k=3)
    centroids = km_tmp.espi_init(data)

    # PCA for visualization
    pca = PCA(n_components=2)
    data_2d = pca.fit_transform(data)
    cent_2d = pca.transform(centroids)

    # Color by dominant-dim value
    sc = ax.scatter(data_2d[:, 0], data_2d[:, 1], c=data[:, dd],
                    cmap='viridis', s=15, alpha=0.6)
    for j in range(3):
        ax.scatter(cent_2d[j, 0], cent_2d[j, 1], marker='*', s=300,
                   color=plt.cm.Set1(j), edgecolors='black', linewidths=1, zorder=5)

    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(f"{name}\nColor = {fnames[dd]} (dominant dim)")
    plt.colorbar(sc, ax=ax, shrink=0.7, label=fnames[dd])

fig.suptitle('ESPI Centroids (stars) Projected via PCA onto Data Cloud', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### Step 4: Initial Full Assignment

After placing the centroids, **every point** is assigned to its nearest centroid using Euclidean distance in the full d-dimensional space. This is done **once** before the iterative loop begins.

**Complexity:** O(n * k * d) — each of the n points is compared to all k centroids.

In [ ]:
# ── Iris (k=3): Before vs After initial assignment ─────────────────
km_iris = KMeansCustom(k=3)
centroids_init = km_iris.espi_init(data_iris)
km_iris.centroids = centroids_init.copy()
labels_init = km_iris.full_assignment(data_iris)
km_iris.labels = labels_init
sse_init = km_iris.compute_sse(data_iris)

pca_iris = PCA(n_components=2)
data_iris_2d = pca_iris.fit_transform(data_iris)
cent_init_2d = pca_iris.transform(centroids_init)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: centroids only, no cluster colors
ax1.scatter(data_iris_2d[:, 0], data_iris_2d[:, 1], c='gray', s=15, alpha=0.5)
for j in range(3):
    ax1.scatter(cent_init_2d[j, 0], cent_init_2d[j, 1], marker='*', s=300,
               color=plt.cm.Set1(j), edgecolors='black', linewidths=1, zorder=5)
ax1.set_title('Before assignment\n(centroids placed, no labels yet)')
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')

# Right: colored by cluster assignment
for j in range(3):
    mask = labels_init == j
    ax2.scatter(data_iris_2d[mask, 0], data_iris_2d[mask, 1],
               c=[plt.cm.Set1(j)], s=15, alpha=0.6, label=f'Cluster {j}')
    ax2.scatter(cent_init_2d[j, 0], cent_init_2d[j, 1], marker='*', s=300,
               color=plt.cm.Set1(j), edgecolors='black', linewidths=1, zorder=5)
ax2.set_title(f'After initial full assignment\nSSE = {sse_init:.2f}')
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
ax2.legend(fontsize=8)

fig.suptitle('Iris — ESPI Initialization: Step 4 (Initial Full Assignment)', fontsize=12)
plt.tight_layout()
plt.show()

## Phase B: Drift-Based Iterative Refinement

After initialization, the algorithm enters an iterative loop that refines the cluster assignments. Each iteration performs four steps:

1. **Update centroids** — recompute each centroid as the mean of its current cluster members.
2. **Compute drift** — measure how much each centroid moved (Euclidean distance between old and new position).
3. **Identify active points** — using the triangle inequality, only points within `2 * drift` of their centroid are near enough to a cluster boundary to potentially switch. All other points are provably stable and **skipped**.
4. **Reassign only active points** — compute full distances only for these boundary points.

**Convergence:** The loop stops when either:
- No point was reassigned in this iteration, or
- The maximum centroid drift falls below epsilon (1e-4).

### How Drift Skipping Works (Triangle Inequality)

If centroid C moved by distance **delta** in one iteration, then for any point P, the triangle inequality tells us:

$$|d(P, C_{\text{new}}) - d(P, C_{\text{old}})| \leq \delta$$

A point that is **deep inside** its cluster (i.e., `dist(P, C) >> delta`) cannot possibly switch to another centroid when the centroids only moved by a small amount.

The algorithm marks a point as **active** (needs rechecking) only if:

$$d(P, C_{\text{own}}) \leq 2 \times \text{drift}[\text{own\_cluster}] + \epsilon$$

This is the set of "boundary" points that are close enough to potentially switch. In later iterations, as centroids stabilize and drift shrinks, **fewer and fewer points** need rechecking, making each iteration progressively cheaper.

In [ ]:
def fit_with_history(data, k, max_iter=100, epsilon=1e-4):
    """
    Run KMeansCustom step-by-step, recording detailed state at each iteration.
    Returns a list of dicts, one per iteration.
    """
    km = KMeansCustom(k=k, max_iterations=max_iter, epsilon=epsilon)
    history = []

    # Phase A: ESPI init
    km.centroids = km.espi_init(data)
    km.labels = km.full_assignment(data)
    init_sse = km.compute_sse(data)

    # Record initial state (iteration 0 = right after init, before any update)
    history.append({
        'iteration': 0,
        'centroids': km.centroids.copy(),
        'old_centroids': km.centroids.copy(),
        'labels': km.labels.copy(),
        'active_indices': list(range(len(data))),  # all points were assigned
        'drift': np.zeros(k),
        'max_drift': 0.0,
        'sse': init_sse,
        'reassigned': len(data),  # initial full assignment
    })

    # Phase B: Iterative refinement
    for it in range(1, max_iter + 1):
        old_centroids = km.centroids.copy()
        km.centroids = km.update_centroids(data)
        drift = km.compute_drift(old_centroids, km.centroids)
        active_indices = km.get_active_points(data, drift)

        reassigned = 0
        for idx in active_indices:
            point = data[idx]
            distances = [km.compute_distance(point, c) for c in km.centroids]
            new_label = np.argmin(distances)
            if new_label != km.labels[idx]:
                km.labels[idx] = new_label
                reassigned += 1

        sse = km.compute_sse(data)
        max_drift = float(np.max(drift))

        history.append({
            'iteration': it,
            'centroids': km.centroids.copy(),
            'old_centroids': old_centroids.copy(),
            'labels': km.labels.copy(),
            'active_indices': list(active_indices),
            'drift': drift.copy(),
            'max_drift': max_drift,
            'sse': sse,
            'reassigned': reassigned,
        })

        if reassigned == 0 or max_drift < epsilon:
            break

    return history

In [ ]:
# ── Iris (k=3): Iteration-by-iteration visualization ───────────────
history_iris = fit_with_history(data_iris, k=3)
pca_iris = PCA(n_components=2)
data_iris_2d = pca_iris.fit_transform(data_iris)

# Select up to 6 key iterations
n_iters = len(history_iris)
if n_iters <= 6:
    sel_indices = list(range(n_iters))
else:
    mid = n_iters // 2
    sel_indices = sorted(set([0, 1, 2, mid, n_iters - 2, n_iters - 1]))

n_plots = len(sel_indices)
ncols = min(3, n_plots)
nrows = (n_plots + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
if n_plots == 1:
    axes = np.array([axes])
axes = np.atleast_2d(axes)

for plot_idx, hist_idx in enumerate(sel_indices):
    row, col = divmod(plot_idx, ncols)
    ax = axes[row, col]
    h = history_iris[hist_idx]

    labels = h['labels']
    centroids_2d = pca_iris.transform(h['centroids'])
    old_cent_2d = pca_iris.transform(h['old_centroids'])
    active = set(h['active_indices'])

    # Plot points colored by cluster
    for j in range(3):
        mask = labels == j
        ax.scatter(data_iris_2d[mask, 0], data_iris_2d[mask, 1],
                   c=[plt.cm.Set1(j)], s=12, alpha=0.5)

    # Highlight active points with black rings
    if h['iteration'] > 0:
        active_mask = np.array([i in active for i in range(len(data_iris))])
        if active_mask.any():
            ax.scatter(data_iris_2d[active_mask, 0], data_iris_2d[active_mask, 1],
                       facecolors='none', edgecolors='black', s=40, linewidths=0.8,
                       zorder=4, label='Active')

    # Centroid stars
    for j in range(3):
        ax.scatter(centroids_2d[j, 0], centroids_2d[j, 1], marker='*', s=250,
                   color=plt.cm.Set1(j), edgecolors='black', linewidths=1, zorder=6)

    # Drift arrows (dashed)
    if h['iteration'] > 0:
        for j in range(3):
            dx = centroids_2d[j, 0] - old_cent_2d[j, 0]
            dy = centroids_2d[j, 1] - old_cent_2d[j, 1]
            if np.sqrt(dx**2 + dy**2) > 0.01:
                ax.annotate('', xy=(centroids_2d[j, 0], centroids_2d[j, 1]),
                            xytext=(old_cent_2d[j, 0], old_cent_2d[j, 1]),
                            arrowprops=dict(arrowstyle='->', color='black',
                                            linestyle='dashed', lw=1.2))

    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(f"Iter {h['iteration']} | SSE={h['sse']:.2f}\n"
                 f"Drift={h['max_drift']:.4f} | Reassign={h['reassigned']} | "
                 f"Active={len(h['active_indices'])}")

# Hide unused subplots
for idx in range(n_plots, nrows * ncols):
    row, col = divmod(idx, ncols)
    axes[row, col].set_visible(False)

fig.suptitle('Iris — Drift-Based Iterative Refinement (PCA 2D)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Convergence Metrics

The algorithm tracks three metrics per iteration:

| Metric | Meaning |
|--------|---------|
| **SSE** (Sum of Squared Errors) | Clustering quality — lower is better |
| **Active / Reassigned points** | Efficiency — fewer means more work is skipped |
| **Max centroid drift** | Stability — smaller means centroids are converging |

Convergence is fast because drift skipping reduces work **exponentially** as centroids stabilize: once drift is small, almost all points are deep inside their clusters and are skipped entirely.

In [ ]:
# ── Convergence metrics for all 3 datasets ─────────────────────────
all_histories = {}
for name, d in datasets.items():
    all_histories[name] = fit_with_history(d['data'], k=3)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for col, (name, hist) in enumerate(all_histories.items()):
    iters = [h['iteration'] for h in hist]
    sses = [h['sse'] for h in hist]
    actives = [len(h['active_indices']) for h in hist]
    reassigns = [h['reassigned'] for h in hist]
    drifts = [h['max_drift'] for h in hist]

    # Row 1: SSE
    ax1 = axes[0, col]
    ax1.plot(iters, sses, 'o-', color='#d62728', markersize=4, linewidth=1.5)
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('SSE')
    ax1.set_title(f'{name} — SSE vs Iteration')
    ax1.grid(True, alpha=0.3)

    # Row 2: Active + Reassigned bars, Drift line on twin axis
    ax2 = axes[1, col]
    x = np.arange(len(iters))
    width = 0.35
    ax2.bar(x - width/2, actives, width, color='#1f77b4', alpha=0.8, label='Active')
    ax2.bar(x + width/2, reassigns, width, color='#ff7f0e', alpha=0.8, label='Reassigned')
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Point count')
    ax2.set_xticks(x)
    ax2.set_xticklabels(iters)
    ax2.legend(loc='upper left', fontsize=7)

    ax2b = ax2.twinx()
    ax2b.plot(x, drifts, 's--', color='#9467bd', markersize=4, linewidth=1.5, label='Max drift')
    ax2b.set_ylabel('Max drift', color='#9467bd')
    ax2b.tick_params(axis='y', labelcolor='#9467bd')
    ax2b.legend(loc='upper right', fontsize=7)

    ax2.set_title(f'{name} — Active/Reassigned + Drift')

fig.suptitle('Convergence Metrics Across All Datasets', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Full Walkthrough on All Datasets

Below we show the complete pipeline for each dataset: the 4-step ESPI initialization (raw data, dominant-dimension heatmap, range segmentation with centroid placement, and initial assignment) followed by selected iteration snapshots showing cluster evolution.

In [ ]:
# ── Full walkthrough for each dataset ──────────────────────────────
for ds_name, ds in datasets.items():
    data = ds['data']
    fnames = ds['names']
    k = 3

    variances = np.var(data, axis=0)
    dd = int(np.argmax(variances))
    d_min_val = data[:, dd].min()
    d_max_val = data[:, dd].max()
    seg_width = (d_max_val - d_min_val) / k
    g_mean = data.mean(axis=0)

    # Build ESPI centroids
    km = KMeansCustom(k=k)
    centroids_espi = km.espi_init(data)
    km.centroids = centroids_espi.copy()
    labels_init_ds = km.full_assignment(data)
    km.labels = labels_init_ds

    # PCA
    pca = PCA(n_components=2)
    data_2d = pca.fit_transform(data)
    cent_2d = pca.transform(centroids_espi)

    # History
    hist = all_histories[ds_name]
    n_h = len(hist)
    if n_h <= 4:
        iter_sel = list(range(n_h))
    else:
        mid_h = n_h // 2
        iter_sel = sorted(set([0, 1, mid_h, n_h - 1]))
    iter_sel = iter_sel[:4]  # max 4 for row 2

    fig, axes = plt.subplots(2, 4, figsize=(18, 8))

    # ── Row 1: ESPI init steps ────────────────────────────
    # (a) Raw data
    ax = axes[0, 0]
    ax.scatter(data_2d[:, 0], data_2d[:, 1], c='gray', s=10, alpha=0.5)
    ax.set_title('(a) Raw data (PCA 2D)')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

    # (b) Dominant-dim heatmap
    ax = axes[0, 1]
    sc = ax.scatter(data_2d[:, 0], data_2d[:, 1], c=data[:, dd],
                    cmap='viridis', s=10, alpha=0.6)
    ax.set_title(f'(b) Colored by {fnames[dd]}\n(dominant dim)')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    plt.colorbar(sc, ax=ax, shrink=0.7)

    # (c) Range segmentation + centroid placement
    ax = axes[0, 2]
    # Use dominant dim as x-axis, second-highest-var as y-axis (or PC2)
    var_order = np.argsort(variances)[::-1]
    dim2 = var_order[1] if len(var_order) > 1 else 0
    ax.scatter(data[:, dd], data[:, dim2], c='gray', s=10, alpha=0.4)
    band_cols = ['#fce4ec', '#e8f5e9', '#e3f2fd']
    for j in range(k):
        lo = d_min_val + j * seg_width
        hi = lo + seg_width
        ax.axvspan(lo, hi, alpha=0.3, color=band_cols[j])
        ax.axvline(lo, color='gray', ls='--', lw=0.7)
        # centroid star
        cx = d_min_val + (j + 0.5) * seg_width
        cy = g_mean[dim2]
        ax.scatter(cx, cy, marker='*', s=250, color=plt.cm.Set1(j),
                   edgecolors='black', linewidths=1, zorder=5)
    ax.axvline(d_max_val, color='gray', ls='--', lw=0.7)
    ax.set_title(f'(c) Range segments + centroids\n{fnames[dd]} vs {fnames[dim2]}')
    ax.set_xlabel(fnames[dd]); ax.set_ylabel(fnames[dim2])

    # (d) Initial assignment
    ax = axes[0, 3]
    for j in range(k):
        mask = labels_init_ds == j
        ax.scatter(data_2d[mask, 0], data_2d[mask, 1],
                   c=[plt.cm.Set1(j)], s=10, alpha=0.5, label=f'C{j}')
        ax.scatter(cent_2d[j, 0], cent_2d[j, 1], marker='*', s=250,
                   color=plt.cm.Set1(j), edgecolors='black', linewidths=1, zorder=5)
    ax.set_title(f'(d) Initial assignment\nSSE={km.compute_sse(data):.2f}')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.legend(fontsize=7)

    # ── Row 2: Selected iterations ────────────────────────
    for col_idx, hi in enumerate(iter_sel):
        ax = axes[1, col_idx]
        h = hist[hi]
        labels_h = h['labels']
        cent_h_2d = pca.transform(h['centroids'])
        old_h_2d = pca.transform(h['old_centroids'])
        active_set = set(h['active_indices'])

        for j in range(k):
            mask = labels_h == j
            ax.scatter(data_2d[mask, 0], data_2d[mask, 1],
                       c=[plt.cm.Set1(j)], s=10, alpha=0.5)

        # Active rings
        if h['iteration'] > 0:
            act_mask = np.array([i in active_set for i in range(len(data))])
            if act_mask.any():
                ax.scatter(data_2d[act_mask, 0], data_2d[act_mask, 1],
                           facecolors='none', edgecolors='black', s=30,
                           linewidths=0.6, zorder=4)

        # Centroids + drift arrows
        for j in range(k):
            ax.scatter(cent_h_2d[j, 0], cent_h_2d[j, 1], marker='*', s=200,
                       color=plt.cm.Set1(j), edgecolors='black', linewidths=1, zorder=6)
            if h['iteration'] > 0:
                dx = cent_h_2d[j, 0] - old_h_2d[j, 0]
                dy = cent_h_2d[j, 1] - old_h_2d[j, 1]
                if np.sqrt(dx**2 + dy**2) > 0.01:
                    ax.annotate('', xy=(cent_h_2d[j, 0], cent_h_2d[j, 1]),
                                xytext=(old_h_2d[j, 0], old_h_2d[j, 1]),
                                arrowprops=dict(arrowstyle='->', color='black',
                                                linestyle='dashed', lw=1))

        ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
        ax.set_title(f"Iter {h['iteration']} | SSE={h['sse']:.2f}\n"
                     f"Drift={h['max_drift']:.4f} | Re={h['reassigned']} | "
                     f"Act={len(h['active_indices'])}")

    # Hide unused row-2 subplots
    for col_idx in range(len(iter_sel), 4):
        axes[1, col_idx].set_visible(False)

    fig.suptitle(f'{ds_name} — Complete ESPI + Iteration Walkthrough', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## Summary

**ESPI Initialization** is O(n * d) and fully deterministic:
- One pass to compute per-feature variances and the global mean.
- Centroids are placed at equally-spaced midpoints along the dominant-variance dimension, with global-mean values on all other dimensions.
- No sorting, no random restarts, no risk of duplicate centroids.

**Drift-Based Iteration** skips stable points, reducing distance computations:
- After each centroid update, the drift (distance moved) determines a radius around each centroid.
- Only points within `2 * drift` of their centroid are near a boundary and need rechecking (triangle inequality guarantee).
- All other points are provably stable and skipped entirely.

**The algorithm converges quickly because:**
1. Good initial centroid spread (ESPI) reduces early-iteration reassignments — centroids start well-separated.
2. Drift shrinks as centroids stabilize, causing fewer active points each iteration — work decreases exponentially.
3. The combination means most iterations only recheck a small fraction of the data, making the overall runtime significantly lower than naive KMeans.